In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


In [2]:
df = pd.read_csv(r'VSRR_Provisional_Drug_Overdose_Death_Counts.csv')

In [3]:
df = df[~df['Indicator'].isin(['Number of Deaths', 'Percent with drugs specified', 'Number of Drug Overdose Deaths'])]

In [4]:
df['Indicator'] = df['Indicator'].replace({
    'Methadone (T40.3)':'Methadone',
    'Natural, semi-synthetic, & synthetic opioids, incl. methadone (T40.2-T40.4)':'Methadone',
    'Natural & semi-synthetic opioids, incl. methadone (T40.2, T40.3)':'Methadone'
})

df['Indicator'] = df['Indicator'].replace({
    'Natural & semi-synthetic opioids (T40.2)':'Opioids',
    'Opioids (T40.0-T40.4,T40.6)':'Opioids',
    'Synthetic opioids, excl. methadone (T40.4)':'Opioids'
})

df['Indicator'] = df['Indicator'].replace({
    'Cocaine (T40.5)':'Cocaine',
    'Heroin (T40.1)':'Heroin',
    'Psychostimulants with abuse potential (T43.6)':'Psychostimulants'
})

In [5]:
df.drop(['Data Value', 'Predicted Value', 'Percent Complete', 'Footnote Symbol', 'Period'], axis=1, inplace=True)

In [6]:
df['Month'] = pd.to_datetime(df['Month'], format='%B').dt.month

In [7]:
X = df.drop(['Indicator'], axis=1)
y = df['Indicator']

In [8]:
df.head()

,State,Year,Month,Indicator,Percent Pending Investigation,State Name,Footnote
0,AK,2015,1,Cocaine,0.0,Alaska,Numbers may differ from published reports usin...
1,AK,2015,2,Cocaine,0.0,Alaska,Numbers may differ from published reports usin...
2,AK,2015,3,Cocaine,0.0,Alaska,Numbers may differ from published reports usin...
3,AK,2015,4,Cocaine,0.0,Alaska,Numbers may differ from published reports usin...
4,AK,2015,5,Cocaine,0.0,Alaska,Numbers may differ from published reports usin...


In [9]:
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ],
    remainder='passthrough'
)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Built an LR Model and got the following results.

LR Accuracy: 0.32911174291375894
                  precision    recall  f1-score   support

         Cocaine       0.00      0.00      0.00      1294
          Heroin       0.40      0.03      0.05      1257
       Methadone       0.31      0.37      0.34      3855
         Opioids       0.34      0.60      0.43      3911
Psychostimulants       0.00      0.00      0.00      1290

        accuracy                           0.33     11607
       macro avg       0.21      0.20      0.17     11607
    weighted avg       0.26      0.33      0.26     11607

I decided to not pursue the logistic regression model further due to it completely ignoring Cocaine and Heroin.

In [12]:
rf_model = RandomForestClassifier()

In [13]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', rf_model)
])